In [3]:
import os
import re
import glob
import argparse
import requests
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from urllib.parse import urlencode
from datetime import datetime
from rasterstats import zonal_stats

In [2]:
def convert_units(value, dataset):
    """Convert rainfall mm→inches and temperature °C→°F."""
    if value is None or np.isnan(value):
        return np.nan
    if dataset == "rainfall":
        return value / 25.4  # mm → inches
    elif dataset == "temperature":
        return (value * 9/5) + 32  # °C → °F
    return value

In [ ]:
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/rf_all/annual"


all_records = []
for tif in sorted(glob.glob(os.path.join(tif_path, f"*.tif"))):
    parts = os.path.basename(tif).replace(".tif", "").split("_")
    year = parts[1]
    date = f"{year}"

    with rasterio.open(tif) as src:
        arr = src.read(1).astype(float)
        arr = np.where(arr == src.nodata, np.nan, arr)
        mean_val = np.nanmean(arr)

    all_records.append({
        "date": date,
        "year": int(year),
        "mean": mean_val
    })

df = pd.DataFrame(all_records)

df["rank"] = df["mean"].rank(method="min", ascending=True)

df

,date,year,mean,rank
0,1920,1920,1546.811332,33.0
1,1921,1921,1904.286918,75.0
2,1922,1922,1799.935181,68.0
3,1923,1923,2512.253361,105.0
4,1924,1924,1638.628186,47.0
...,...,...,...,...
101,2021,2021,1778.105613,63.0
102,2022,2022,1128.450986,4.0
103,2023,2023,1589.244796,41.0
104,2024,2024,1419.567413,20.0


In [121]:
#AUGUST RAIN

tif_path = "/Users/cherryleheu/Documents/HCDP/Data/rf_all/monthly"

for m in np.arange(1,13):
    all_records = []
    for tif in sorted(glob.glob(os.path.join(tif_path, f"*{m:02d}.tif"))):
        parts = os.path.basename(tif).replace(".tif", "").split("_")
        year = parts[1]
        date = f"{year}"

        with rasterio.open(tif) as src:
            arr = src.read(1).astype(float)
            arr = np.where(arr == src.nodata, np.nan, arr)
            mean_val = np.nanmean(arr)

        all_records.append({
            "date": date,
            "year": int(year),
            "mean": mean_val
        })

    df = pd.DataFrame(all_records)

    df["rank"] = df["mean"].rank(method="min", ascending=True)

    print(f"For month: {m}: {df[df['date']=='2025']['rank'].values}")


For month: 1: [56.]
For month: 2: [4.]
For month: 3: [31.]
For month: 4: [25.]
For month: 5: [21.]
For month: 6: [51.]
For month: 7: [16.]
For month: 8: [3.]
For month: 9: [23.]
For month: 10: [21.]
For month: 11: [40.]
For month: 12: [14.]


In [ ]:
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/rf_all/monthly"

all_records = []
for tif in sorted(glob.glob(os.path.join(tif_path, f"*08.tif"))):
    parts = os.path.basename(tif).replace(".tif", "").split("_")
print(parts)

['rainfall', '2025', '08']


In [132]:
#Tmean
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/annual/tmean/calendar_year"

all_records = []

with rasterio.open('/Users/cherryleheu/Documents/HCDP/Data/climo/30yr/monthly_tmean_statewide_1991-2020_annual.tif') as src:
    clim = src.read(1, masked=True).astype("float64")
    climo_mean = clim.mean()
    climo_f = climo_mean*9/5 + 32

for tif in sorted(glob.glob(os.path.join(tif_path, f"*_agg.tif"))):
    parts = os.path.basename(tif).replace(".tif", "").split("_")
    year = parts[2]
    date = f"{year}"

    with rasterio.open(tif) as src:
        arr = src.read(1).astype(float)
        arr = np.where(arr == src.nodata, np.nan, arr)
        mean_val = np.nanmean(arr)
        mean_val_f = mean_val*9/5 + 32

    # anom = mean_val*9/5 + 32
    all_records.append({
        "date": date,
        "year": int(year),
        "mean": mean_val_f,
        "anom": mean_val_f - climo_f
    })

df = pd.DataFrame(all_records)

df["rank"] = df["mean"].rank(method="min", ascending=False)

df

,date,year,mean,anom,rank
0,1990,1990,65.801174,-0.492514,28.0
1,1991,1991,65.641067,-0.652622,32.0
2,1992,1992,65.888010,-0.405678,27.0
3,1993,1993,65.107939,-1.185750,35.0
4,1994,1994,65.549973,-0.743716,33.0
5,1995,1995,66.172545,-0.121143,22.0
6,1996,1996,66.240123,-0.053566,20.0
7,1997,1997,65.703645,-0.590043,30.0
8,1998,1998,65.045232,-1.248457,36.0
9,1999,1999,65.273941,-1.019748,34.0


In [131]:
import numpy as np
import rasterio

path = "/Users/cherryleheu/Documents/HCDP/Data/climo/30yr/monthly_tmean_statewide_1991-2020_annual.tif"

with rasterio.open(path) as src:
    clim = src.read(1, masked=True).astype("float64")  # respects nodata if present
    climo_mean_c = float(clim.mean())                  # masked mean ignores nodata
    climo_mean_f = climo_mean_c * 9/5 + 32

print("climo_mean (C):", climo_mean_c)
print("climo_mean (F):", climo_mean_f)


climo_mean (C): 19.05204927716812
climo_mean (F): 66.29368869890261


In [ ]:
#County
shapefile = "../../public/shapefiles/islands.shp"
# raster_folder = "/Users/cherryleheu/Documents/HCDP/Data/monthly/rainfall/"
climo_file = "/Users/cherryleheu/Documents/HCDP/Data/climo/30yr/monthly_rainfall_clim_statewide_1991-2020_annual.tif"
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/rf_all"
gdf = gpd.read_file(shapefile).dissolve(by="name", as_index=False).reset_index(drop=True)

climo_stats = zonal_stats(gdf, climo_file, stats=["mean"])
gdf["climo_mean"] = [(c["mean"] / 25.4) if c["mean"] is not None else np.nan for c in climo_stats]

all_records = []

for tif in sorted(glob.glob(os.path.join(tif_path, "*.tif"))):
    name = os.path.basename(tif).replace(".tif", "")
    parts = name.split("_")
    year = parts[1] 

    with rasterio.open(tif) as src:
        nodata = src.nodata

    stats = zonal_stats(gdf, tif, stats=["mean"], nodata=nodata)

    for i, division in gdf.iterrows():
        mean_raw = stats[i]["mean"]
        climo_mean = division["climo_mean"]

        if mean_raw is None or climo_mean is None or climo_mean == 0 or np.isnan(climo_mean):
            mean_val = np.nan
            anomaly = np.nan
        else:
            mean_val = mean_raw / 25.4
            anomaly = (mean_val - climo_mean) / climo_mean * 100

        all_records.append({
            "island": division["name"],
            "date": int(year),
            "mean": mean_val,
            "value": f"{mean_val:.2f} ({anomaly:+.2f})",
        })

df = pd.DataFrame(all_records)
df["rank"] = df.groupby("island")["mean"].rank(method="min", ascending=True).astype("Int64")
print(df[df['date'] == 2025])
df[df['date'] == 2025].to_csv("island_annual_rainfall_2025_ranks.csv", index=False)

        island  date       mean           value  rank
735     Hawaii  2025  40.653415  40.65 (-36.07)     2
736  Kahoolawe  2025   9.918075   9.92 (-14.39)    23
737      Kauai  2025  67.016988  67.02 (-11.12)    21
738      Lanai  2025  12.761269  12.76 (-28.00)    15
739       Maui  2025  40.665760  40.67 (-40.66)     1
740    Molokai  2025  25.554383  25.55 (-38.13)     5
741       Oahu  2025  46.042890  46.04 (-15.08)    16


In [30]:
#County
shapefile = "../../public/shapefiles/islands.shp"
# raster_folder = "/Users/cherryleheu/Documents/HCDP/Data/monthly/rainfall/"
climo_file = "/Users/cherryleheu/Documents/HCDP/Data/climo/30yr/monthly_rainfall_clim_statewide_1991-2020_annual.tif"
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/rf_all/annual"
gdf = gpd.read_file(shapefile).dissolve(by="name", as_index=False).reset_index(drop=True)

climo_stats = zonal_stats(gdf, climo_file, stats=["mean"])
gdf["climo_mean"] = [(c["mean"] / 25.4) if c["mean"] is not None else np.nan for c in climo_stats]

all_records = []

for tif in sorted(glob.glob(os.path.join(tif_path, "*.tif"))):
    name = os.path.basename(tif).replace(".tif", "")
    parts = name.split("_")
    year = parts[1] 
    with rasterio.open(tif) as src:
        nodata = src.nodata

    stats = zonal_stats(gdf, tif, stats=["mean"], nodata=nodata)

    for i, division in gdf.iterrows():
        mean_raw = stats[i]["mean"]
        climo_mean = division["climo_mean"]

        if mean_raw is None or climo_mean is None or climo_mean == 0 or np.isnan(climo_mean):
            mean_val = np.nan
            anomaly = np.nan
        else:
            mean_val = mean_raw / 25.4
            anomaly = (mean_val - climo_mean) / climo_mean * 100

        all_records.append({
            "island": division["name"],
            "date": int(year),
            "mean": mean_val,
            "p_anomaly": f"{round(anomaly)}%",
            "diff_anomaly": mean_val - climo_mean,
        })

df = pd.DataFrame(all_records)
df["rank"] = df.groupby("island")["mean"].rank(method="min", ascending=True).astype("Int64")
print(df[df['date'] == 2025])
df[df['date'] == 2025].to_csv("island_annual_rainfall_2025_ranks.csv", index=False)

        island  date       mean p_anomaly  diff_anomaly  rank
735     Hawaii  2025  40.653415      -36%    -22.937288     2
736  Kahoolawe  2025   9.918075      -14%     -1.666635    23
737      Kauai  2025  67.016988      -11%     -8.386554    21
738      Lanai  2025  12.761269      -28%     -4.961560    15
739       Maui  2025  40.665760      -41%    -27.868129     1
740    Molokai  2025  25.554383      -38%    -15.751622     5
741       Oahu  2025  46.042890      -15%     -8.173708    16


In [26]:
df["rank"] = df.groupby("island")["mean"].rank(method="min", ascending=True).astype("Int64")
print(df[df['date'] == 2025])

        island  date       mean p_anomaly  diff_anomaly           value  rank
735     Hawaii  2025  40.653415   -36.07%    -22.937288  40.65 (-36.07)     2
736  Kahoolawe  2025   9.918075   -14.39%     -1.666635   9.92 (-14.39)    23
737      Kauai  2025  67.016988   -11.12%     -8.386554  67.02 (-11.12)    21
738      Lanai  2025  12.761269   -28.00%     -4.961560  12.76 (-28.00)    15
739       Maui  2025  40.665760   -40.66%    -27.868129  40.67 (-40.66)     1
740    Molokai  2025  25.554383   -38.13%    -15.751622  25.55 (-38.13)     5
741       Oahu  2025  46.042890   -15.08%     -8.173708  46.04 (-15.08)    16


In [ ]:
#County
shapefile = "../../public/shapefiles/islands.shp"
# raster_folder = "/Users/cherryleheu/Documents/HCDP/Data/monthly/rainfall/"
climo_file = "/Users/cherryleheu/Documents/HCDP/Data/climo/30yr/monthly_tmean_statewide_1991-2020_annual.tif"
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/annual/tmean/calendar_year"
gdf = gpd.read_file(shapefile).dissolve(by="name", as_index=False).reset_index(drop=True)

climo_stats = zonal_stats(gdf, climo_file, stats=["mean"])
gdf["climo_mean"] = [(c["mean"] * 1.8 + 32) if c["mean"] is not None else np.nan for c in climo_stats]

all_records = []

for tif in sorted(glob.glob(os.path.join(tif_path, "*_agg.tif"))):
    name = os.path.basename(tif).replace(".tif", "")
    parts = name.split("_")
    year = parts[2] 

    with rasterio.open(tif) as src:
        nodata = src.nodata

    stats = zonal_stats(gdf, tif, stats=["mean"], nodata=nodata)

    for i, division in gdf.iterrows():
        mean_raw = stats[i]["mean"]
        climo_mean = division["climo_mean"]

        if mean_raw is None or climo_mean is None or climo_mean == 0 or np.isnan(climo_mean):
            mean_val = np.nan
            anomaly = np.nan
        else:
            mean_val = mean_raw * 1.8 + 32
            anomaly = (mean_val - climo_mean) 

        all_records.append({
            "island": division["name"],
            "date": int(year),
            "mean": mean_val,
            "value": f"{mean_val:.2f} ({anomaly:+.2f})",
        })

tmeandf = pd.DataFrame(all_records)
tmeandf["rank"] = tmeandf.groupby("island")["mean"].rank(method="min", ascending=False).astype("Int64")

print(tmeandf[tmeandf['date'] == 2025])
tmeandf[tmeandf['date'] == 2025].to_csv("island_annual_tmean_2025_tmean_ranks.csv", index=False)

        island  date       mean          value  rank
245     Hawaii  2025  63.975182  63.98 (+0.69)     8
246  Kahoolawe  2025  74.937319  74.94 (+0.89)     4
247      Kauai  2025  71.795279  71.80 (+0.91)     3
248      Lanai  2025  73.490867  73.49 (+0.86)     4
249       Maui  2025  69.666219  69.67 (+0.78)     3
250    Molokai  2025  73.468817  73.47 (+0.86)     4
251       Oahu  2025  74.491756  74.49 (+1.01)     4


In [ ]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask

# inputs
islands_shp = "../../public/islands.shp"
island_col = "name"
tif = "/Users/cherryleheu/Documents/HCDP/Data/annual/tmean/calendar_year/annual_tmean_2025_agg.tif"

# dissolve to island polygons
gdf = gpd.read_file(islands_shp).dissolve(by=island_col, as_index=False)

with rasterio.open(tif) as src:
    arr = src.read(1).astype("float64")
    nodata = src.nodata
    transform = src.transform
    crs = src.crs

# match CRS
gdf = gdf.to_crs(crs)

# nodata mask
valid = np.isfinite(arr)
if nodata is not None:
    valid &= (arr != nodata)

island_stats = []
total_sum = 0.0
total_n = 0

for _, row in gdf.iterrows():
    geom = row.geometry

    # mask True outside the island; False inside
    outside = geometry_mask([geom], transform=transform, invert=False, out_shape=arr.shape)

    inside_valid = (~outside) & valid

    n = int(inside_valid.sum())
    s = float(arr[inside_valid].sum()) if n > 0 else 0.0

    island_stats.append({"island": row[island_col], "pix_n": n, "pix_sum": s})

    total_sum += s
    total_n += n

statewide_mean = total_sum / total_n if total_n > 0 else np.nan

print("Statewide mean (pixel-weighted):", statewide_mean)


Statewide mean (pixel-weighted): 19.47679762513816


In [ ]:
#Tmin
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/annual/tmin/calendar_year"

all_records = []
for tif in sorted(glob.glob(os.path.join(tif_path, f"*_agg.tif"))):
    parts = os.path.basename(tif).replace(".tif", "").split("_")
    year = parts[2]
    date = f"{year}"

    with rasterio.open(tif) as src:
        arr = src.read(1).astype(float)
        arr = np.where(arr == src.nodata, np.nan, arr)
        mean_val = np.nanmean(arr)

    all_records.append({
        "date": date,
        "year": int(year),
        "mean": mean_val*9/5 + 32
    })

df = pd.DataFrame(all_records)

df["rank"] = df["mean"].rank(method="min", ascending=False)

df

,date,year,mean,rank
0,1990,1990,58.256866,29.0
1,1991,1991,57.645122,35.0
2,1992,1992,58.279095,28.0
3,1993,1993,57.592039,36.0
4,1994,1994,58.533491,23.0
5,1995,1995,58.372302,26.0
6,1996,1996,59.096838,14.0
7,1997,1997,58.538588,22.0
8,1998,1998,57.696056,33.0
9,1999,1999,57.660617,34.0


In [ ]:
#Tmax
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/annual/tmax/calendar_year"

all_records = []
for tif in sorted(glob.glob(os.path.join(tif_path, f"*_agg.tif"))):
    parts = os.path.basename(tif).replace(".tif", "").split("_")
    year = parts[2]
    date = f"{year}"

    with rasterio.open(tif) as src:
        arr = src.read(1).astype(float)
        arr = np.where(arr == src.nodata, np.nan, arr)
        mean_val = np.nanmean(arr)

    all_records.append({
        "date": date,
        "year": int(year),
        "mean": mean_val*9/5 + 32
    })

df = pd.DataFrame(all_records)

df["rank"] = df["mean"].rank(method="min", ascending=False)

df

,date,year,mean,rank
0,1990,1990,73.345689,30.0
1,1991,1991,73.637233,23.0
2,1992,1992,73.497341,25.0
3,1993,1993,72.623943,34.0
4,1994,1994,72.566764,35.0
5,1995,1995,73.973006,17.0
6,1996,1996,73.383554,29.0
7,1997,1997,72.869085,33.0
8,1998,1998,72.394630,36.0
9,1999,1999,72.887622,32.0
